In [4]:
import pandas as pd


# Upload Urban Growth CSV
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "outputs" / "steps1-4"
OUT.mkdir(parents=True, exist_ok=True)
uploaded = DATA / "EarthLens_Urban_Growth_Analysis.csv"


df = pd.read_csv(uploaded)

# Prepare annual time series
annual = df[df["Record_Type"] == "annual_growth"].copy()
annual["Year"] = annual["Year"].astype(int)
annual = annual.sort_values("Year")

# Prepare monthly time series
monthly = df[df["Record_Type"] == "monthly_growth"].copy()
monthly["Date"] = pd.to_datetime(monthly["Date"])

# Keep only complete years for monthly comparison
monthly = monthly[monthly["Date"].dt.year <= 2025]
monthly = monthly.sort_values("Date")

print("Annual observations:", len(annual))
print("Monthly observations:", len(monthly))

print("\nAnnual Urban Growth:")
print(annual[["Year", "Confident_Built_km2"]])

Annual observations: 9
Monthly observations: 120

Annual Urban Growth:
     Year  Confident_Built_km2
130  2017            53.251762
131  2018            55.883802
132  2019            63.078839
133  2020            75.104223
134  2021            79.395828
135  2022            88.111877
136  2023            98.730016
137  2024           110.710762
138  2025           118.491164


In [12]:

import pandas as pd
import matplotlib.pyplot as plt

# 1. Create a new monthly dataset
monthly_ts = df[
    df["Record_Type"] == "monthly_growth"
].copy()

# 2. Prepare dates
monthly_ts["Date"] = pd.to_datetime(
    monthly_ts["Date"]
)

# 3. Keep complete years only (2016-2025)
monthly_ts = monthly_ts[
    (monthly_ts["Date"] >= "2016-01-01") &
    (monthly_ts["Date"] < "2026-01-01")
].copy()

monthly_ts = monthly_ts.sort_values("Date")

# 4. Check the data
print("Observations:", len(monthly_ts))

print(
    "Missing values:",
    monthly_ts["Confident_Built_km2"].isna().sum()
)

print(
    "Duplicate dates:",
    monthly_ts["Date"].duplicated().sum()
)

# 5. Calculate 12-month moving average
monthly_ts["MA_12"] = (
    monthly_ts["Confident_Built_km2"]
    .rolling(window=12)
    .mean()
)

# 6. Visualize
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    monthly_ts["Date"],
    monthly_ts["Confident_Built_km2"],
    label="Original Monthly Data",
    color="royalblue",
    alpha=0.55
)

ax.plot(
    monthly_ts["Date"],
    monthly_ts["MA_12"],
    label="12-Month Moving Average",
    color="darkorange",
    linewidth=3
)

ax.set_title("EarthLens - Monthly Urban Growth")
ax.set_xlabel("Date")
ax.set_ylabel("Estimated Built-up Area (km²)")

ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()

# 7. Save the chart
fig.savefig(
    "EarthLens_Monthly_Trend.png",
    dpi=300,
    bbox_inches="tight"
)

# 8. Display in Streamlit
plt.show()

plt.close(fig)


2026-11-14 00:30:09.621 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 00:30:09.623 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 00:30:09.625 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 00:30:09.627 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 00:30:09.628 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 00:30:09.630 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 00:30:09.633 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-14 00:30:09.635 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [15]:
"""EarthLens: Load and validate the data, annual trend and monthly time series.

Place this script in the same directory as EarthLens_Urban_Growth_Analysis*.csv.
Run locally with: python EarthLens_Steps_1_to_3.py
"""

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# 1. Find the CSV locally (no Colab or Streamlit needed)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
folder = ROOT / "outputs" / "steps1-4"   # where figures are saved
folder.mkdir(parents=True, exist_ok=True)
matches = list((ROOT / "data").glob("*Urban_Growth_Analysis*.csv"))
if not matches:
    raise FileNotFoundError(
        f"Could not find the Urban Growth CSV in: {folder}\n"
        "Put the CSV in the data/ folder, then run again."
    )

csv_path = max(matches, key=lambda path: path.stat().st_mtime)
print("Reading file:", csv_path.name)
df = pd.read_csv(csv_path)

# 2. Check required columns
required = {"Record_Type", "Date", "Year", "Confident_Built_km2"}
missing_cols = required - set(df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns: {sorted(missing_cols)}")

# 3. Annual data (2017-2025)
annual_ts = df.loc[df["Record_Type"].eq("annual_growth"),
                   ["Year", "Confident_Built_km2"]].copy()
annual_ts["Year"] = pd.to_numeric(annual_ts["Year"], errors="coerce")
annual_ts["Confident_Built_km2"] = pd.to_numeric(
    annual_ts["Confident_Built_km2"], errors="coerce"
)
annual_ts = annual_ts.dropna().sort_values("Year")
annual_ts = annual_ts[annual_ts["Year"].between(2017, 2025)].copy()
annual_ts["Year"] = annual_ts["Year"].astype(int)

if annual_ts.empty:
    raise ValueError("No valid annual observations were found.")

start_area = annual_ts["Confident_Built_km2"].iloc[0]
end_area = annual_ts["Confident_Built_km2"].iloc[-1]
net_change = end_area - start_area
growth_pct = (net_change / start_area) * 100 if start_area != 0 else float("nan")

print("\n--- ANNUAL SUMMARY ---")
print("Annual observations:", len(annual_ts))
print(annual_ts.to_string(index=False))
print(f"Starting built-up estimate: {start_area:.2f} km²")
print(f"Final built-up estimate: {end_area:.2f} km²")
print(f"Net change: {net_change:+.2f} km²")
print(f"Total percentage change: {growth_pct:+.2f}%")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(annual_ts["Year"], annual_ts["Confident_Built_km2"],
        color="royalblue", marker="o", linewidth=2)
ax.set(title="EarthLens - Annual Urban Growth (2017-2025)",
       xlabel="Year", ylabel="Estimated built-up area (km²)")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(folder / "EarthLens_Annual_Trend.png", dpi=300)
plt.show()

# 4. Monthly data (full years 2016-2025 only)
monthly_ts = df.loc[df["Record_Type"].eq("monthly_growth"),
                    ["Date", "Confident_Built_km2"]].copy()
monthly_ts["Date"] = pd.to_datetime(monthly_ts["Date"], errors="coerce")
monthly_ts["Confident_Built_km2"] = pd.to_numeric(
    monthly_ts["Confident_Built_km2"], errors="coerce"
)
monthly_ts = monthly_ts.loc[
    monthly_ts["Date"].between("2016-01-01", "2025-12-31")
].sort_values("Date").copy()

print("\n--- MONTHLY DATA CHECK ---")
print("Monthly observations:", len(monthly_ts))
print("Missing built-up values:", monthly_ts["Confident_Built_km2"].isna().sum())
print("Duplicate dates:", monthly_ts["Date"].duplicated().sum())
print("First date:", monthly_ts["Date"].min().date())
print("Last date:", monthly_ts["Date"].max().date())

if monthly_ts["Confident_Built_km2"].isna().any():
    raise ValueError("Monthly data has missing values; review them before plotting.")
if monthly_ts["Date"].duplicated().any():
    raise ValueError("Monthly data has duplicate dates; review before plotting.")

# Confirm every expected month exists
expected_months = pd.date_range("2016-01-01", "2025-12-01", freq="MS")
missing_months = expected_months.difference(pd.DatetimeIndex(monthly_ts["Date"]))
print("Missing months in 2016-2025:", len(missing_months))
if len(missing_months):
    print("Missing month dates:", missing_months.strftime("%Y-%m").tolist())

# 5. 12-month moving average (calculation does not change original values)
monthly_ts["MA_12"] = (
    monthly_ts["Confident_Built_km2"].rolling(window=12, min_periods=12).mean()
)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(monthly_ts["Date"], monthly_ts["Confident_Built_km2"],
        label="Original monthly estimates", color="royalblue", alpha=0.55)
ax.plot(monthly_ts["Date"], monthly_ts["MA_12"],
        label="12-month moving average", color="darkorange", linewidth=2.5)
ax.set(title="EarthLens - Monthly Urban Growth (2016-2025)",
       xlabel="Date", ylabel="Estimated built-up area (km²)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(folder / "EarthLens_Monthly_Trend.png", dpi=300)
plt.show()

print("\nSaved:")
print(folder / "EarthLens_Annual_Trend.png")
print(folder / "EarthLens_Monthly_Trend.png")
print("Steps 1-3 completed successfully.")

Reading file: EarthLens_Urban_Growth_Analysis__1_.csv

--- ANNUAL SUMMARY ---
Annual observations: 9
 Year  Confident_Built_km2
 2017            53.251762
 2018            55.883802
 2019            63.078839
 2020            75.104223
 2021            79.395828
 2022            88.111877
 2023            98.730016
 2024           110.710762
 2025           118.491164
Starting built-up estimate: 53.25 km²
Final built-up estimate: 118.49 km²
Net change: +65.24 km²
Total percentage change: +122.51%


C:\Users\ASUS\AppData\Local\Temp\ipykernel_6248\1568252454.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



--- MONTHLY DATA CHECK ---
Monthly observations: 120
Missing built-up values: 0
Duplicate dates: 0
First date: 2016-01-01
Last date: 2025-12-01
Missing months in 2016-2025: 0

Saved:
C:\Users\ASUS\Downloads\files (2)\EarthLens_Annual_Trend.png
C:\Users\ASUS\Downloads\files (2)\EarthLens_Monthly_Trend.png
Steps 1-3 completed successfully.


C:\Users\ASUS\AppData\Local\Temp\ipykernel_6248\1568252454.py:117: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
# STEP 4: SEASONAL DECOMPOSITION

from statsmodels.tsa.seasonal import STL
import matplotlib.pyplot as plt

# Use the monthly data from previous steps
ts = monthly_ts.copy()

ts["Date"] = pd.to_datetime(ts["Date"])

ts = ts.set_index("Date").sort_index()

# Ensure regular monthly frequency
ts = ts["Confident_Built_km2"].asfreq("MS")

# Check missing values
if ts.isna().any():
    raise ValueError("Monthly series has missing values!")

# STL Decomposition
stl = STL(
    ts,
    period=12,
    robust=True
)

result = stl.fit()

# Plot four components
fig = result.plot()
fig.set_size_inches(13, 10)

fig.suptitle(
    "EarthLens - Monthly Seasonal Decomposition",
    fontsize=16,
    y=1.01
)

plt.tight_layout()

plt.savefig(
    OUT / "EarthLens_Seasonal_Decomposition.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Save separate components
decomposition_df = pd.DataFrame({
    "Observed": ts,
    "Trend": result.trend,
    "Seasonal": result.seasonal,
    "Residual": result.resid
})

decomposition_df.to_csv(
    OUT / "EarthLens_Decomposition_Results.csv"
)

print("Seasonal decomposition completed!")
print(decomposition_df.head())


Seasonal decomposition completed!
             Observed      Trend   Seasonal  Residual
Date                                                 
2016-01-01  79.848377  48.006104  31.313836  0.528438
2016-02-01  62.410604  48.816142  14.245438 -0.650976
2016-03-01  54.137587  49.620891   5.498470 -0.981775
2016-04-01  37.367151  50.419326 -12.589596 -0.462579
2016-05-01  27.748428  51.210949 -24.119319  0.656797


C:\Users\ASUS\AppData\Local\Temp\ipykernel_6248\964963299.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
